In [ ]:
# Metagenome Assembly Statistics + Merge AMR to CAT (Site A1, ERR4419502)
list.files()

In [ ]:
# 1. Load the data
df <- read.delim("A1_seqtk_comp.tabular.tabular", header = TRUE, sep = "\t", check.names = FALSE)
colnames(df) <- gsub("^#", "", colnames(df))
head(df)

In [ ]:
# 2. Items 1-5: basic contig length statistics
total_contigs <- nrow(df)
min_length    <- min(df$length)
max_length    <- max(df$length)
avg_length    <- mean(df$length)
median_length <- median(df$length)

cat("1. Total number of contigs :", total_contigs, "\n")
cat("2. Minimum contig length   :", min_length, "bp\n")
cat("3. Maximum contig length   :", max_length, "bp\n")
cat("4. Average contig length   :", round(avg_length, 2), "bp\n")
cat("5. Median contig length    :", round(median_length, 2), "bp\n")

In [ ]:
# 3. Item 7: total assembly length
total_assembly_length <- sum(df$length)
cat("7. Total assembly length :", total_assembly_length, "bp\n")

In [ ]:
# 4. Item 8: overall GC content (%) across the whole assembly
total_A <- sum(df$A)
total_C <- sum(df$C)
total_G <- sum(df$G)
total_T <- sum(df$T)

overall_gc_percent <- (total_G + total_C) / (total_A + total_C + total_G + total_T) * 100
cat("8. Overall GC content :", round(overall_gc_percent, 2), "%\n")

In [ ]:
# 5. Item 9: GC content per contig (%) + histogram
df$GC_percent <- (df$G + df$C) / (df$A + df$C + df$G + df$T) * 100

hist(df$GC_percent,
     breaks = 50,
     col = "#16a34a",
     border = "white",
     xlab = "GC content (%)",
     ylab = "Number of contigs",
     main = "GC Content Distribution - Site A1 (ERR4419502)")

In [ ]:
# 6. Item 6: contig length distribution (histogram)
hist(df$length,
     breaks = 50,
     col = "#2563eb",
     border = "white",
     xlab = "Contig length (bp)",
     ylab = "Number of contigs",
     main = "Contig Length Distribution - Site A1 (ERR4419502)")

In [ ]:
# 7. Item 10: number of ambiguous bases (N)
df$N_bases <- df$length - (df$A + df$C + df$G + df$T)
total_N_bases <- sum(df$N_bases)
cat("10. Number of ambiguous bases (N) :", total_N_bases, "\n")

len <- sort(df$length, decreasing = TRUE)
total <- sum(len)
n50 <- len[which(cumsum(len) >= total / 2)[1]]
cat("N50 :", n50, "bp\n")

In [ ]:
# 8. Final summary (all 10 items) + save to file
cat(strrep("=", 55), "\n")
cat(" Metagenome Assembly Statistics (Site A1)\n")
cat("Fulwah Alsahman\n")
cat(strrep("=", 55), "\n")
cat(sprintf("1. Total number of contigs       : %d\n", total_contigs))
cat(sprintf("2. Minimum contig length          : %d bp\n", min_length))
cat(sprintf("3. Maximum contig length          : %d bp\n", max_length))
cat(sprintf("4. Average contig length          : %.2f bp\n", avg_length))
cat(sprintf("5. Median contig length           : %.2f bp\n", median_length))
cat(sprintf("7. Total assembly length          : %d bp\n", total_assembly_length))
cat(sprintf("8. Overall GC content             : %.2f %%\n", overall_gc_percent))
cat(sprintf("10. Number of ambiguous bases (N) : %d\n", total_N_bases))
cat(strrep("=", 55), "\n")

summary_lines <- c(
  " Metagenome Assembly Statistics (Site A1, ERR4419502)",
  " Fulwah Alsahman",
  strrep("=", 55),
  sprintf("1. Total number of contigs       : %d", total_contigs),
  sprintf("2. Minimum contig length          : %d bp", min_length),
  sprintf("3. Maximum contig length          : %d bp", max_length),
  sprintf("4. Average contig length          : %.2f bp", avg_length),
  sprintf("5. Median contig length           : %.2f bp", median_length),
  sprintf("7. Total assembly length          : %d bp", total_assembly_length),
  sprintf("8. Overall GC content             : %.2f %%", overall_gc_percent),
  sprintf("10. Number of ambiguous bases (N) : %d", total_N_bases)
)
writeLines(summary_lines, "assembly_stats_summary.txt")
cat("\nSaved: assembly_stats_summary.txt\n")

In [ ]:
# Code 9: Merge AMRFinderPlus with CAT taxonomy -> Excel file with 4 sheets

install.packages("openxlsx")   # run once
library(openxlsx)

amr <- read.delim("A1_amrfinderplus_report.tabular.tabular",
                  check.names = FALSE)
cat_df <- read.delim("A1_cat_contig_taxonomy.tabular.tabular",
                     check.names = FALSE)

names(cat_df)[names(cat_df) == "# contig"] <- "Contig id"

# Remove confidence scores from CAT names (e.g. "Bacteria: 0.98" -> "Bacteria")
ranks <- c("superkingdom", "phylum", "class", "order", "family", "genus", "species")
for (r in ranks) cat_df[[r]] <- sub(":\\s*[0-9.]+$", "", as.character(cat_df[[r]]))

# Left join on contig ID: keep every AMRFinderPlus row, add CAT taxonomy
m <- merge(amr, cat_df, by = "Contig id", all.x = TRUE, sort = FALSE)
m <- m[order(match(m[["Contig id"]], amr[["Contig id"]])), ]

nrow(amr); nrow(m)   # must be equal

# Sheet 1: AMRFinderPlus_Summary
summ <- data.frame(
  "Contig ID"    = amr[["Contig id"]],
  "Gene"         = amr[["Element symbol"]],
  "Type"         = amr[["Type"]],
  "Subtype"      = amr[["Subtype"]],
  "Class"        = amr[["Class"]],
  "Subclass"     = amr[["Subclass"]],
  "Coverage (%)" = amr[["% Coverage of reference"]],
  "Identity (%)" = amr[["% Identity to reference"]],
  check.names = FALSE
)

# Sheet 2: CAT_Taxonomy_AMR_Contigs
tax <- data.frame(
  "Contig ID"      = m[["Contig id"]],
  "Classification" = m[["classification"]],
  "Reason"         = m[["reason"]],
  "Superkingdom"   = m[["superkingdom"]],
  "Phylum"         = m[["phylum"]],
  "Class"          = m[["class"]],
  "Order"          = m[["order"]],
  "Family"         = m[["family"]],
  "Genus"          = m[["genus"]],
  "Species"        = m[["species"]],
  check.names = FALSE
)

# Sheet 3: Final_Merged
merged <- data.frame(
  "Sample ID"       = "A1",
  "Contig ID"       = m[["Contig id"]],
  "Gene"            = m[["Element symbol"]],
  "Type"            = m[["Type"]],
  "Subtype"         = m[["Subtype"]],
  "Class"           = m[["Class"]],
  "Subclass"        = m[["Subclass"]],
  "Coverage (%)"    = m[["% Coverage of reference"]],
  "Identity (%)"    = m[["% Identity to reference"]],
  "Classification"  = m[["classification"]],
  "Reason"          = m[["reason"]],
  "Superkingdom"    = m[["superkingdom"]],
  "Phylum"          = m[["phylum"]],
  "Taxonomic Class" = m[["class"]],
  "Order"           = m[["order"]],
  "Family"          = m[["family"]],
  "Genus"           = m[["genus"]],
  "Species"         = m[["species"]],
  check.names = FALSE
)

# Sheet 4: AMRFinderPlus_Full_Report (original output, unchanged)
full <- amr

# Write the Excel file
sheets <- list(
  "AMRFinderPlus_Summary"      = summ,
  "CAT_Taxonomy_AMR_Contigs"   = tax,
  "Final_Merged"               = merged,
  "AMRFinderPlus_Full_Report"  = full
)

wb <- createWorkbook()
for (nm in names(sheets)) {
  addWorksheet(wb, nm)
  writeData(wb, nm, sheets[[nm]])
  freezePane(wb, nm, firstRow = TRUE)
  setColWidths(wb, nm, cols = 1:ncol(sheets[[nm]]), widths = "auto")
}
saveWorkbook(wb, "A1_Week4_AMR_Taxonomy.xlsx", overwrite = TRUE)

print(merged)

In [ ]:
# Code 10: AMR-only genes (Type = AMR) with Part C headers -> Excel

amr_only <- merged[merged[["Type"]] == "AMR", ]

amr_only <- data.frame(
  "Sample ID"        = amr_only[["Sample ID"]],
  "Contig ID"        = amr_only[["Contig ID"]],
  "Gene"             = amr_only[["Gene"]],
  "Type"             = amr_only[["Type"]],
  "Subtype"          = amr_only[["Subtype"]],
  "Resistance Class" = amr_only[["Class"]],
  "Subclass"         = amr_only[["Subclass"]],
  "Coverage (%)"     = amr_only[["Coverage (%)"]],
  "Identity (%)"     = amr_only[["Identity (%)"]],
  "Superkingdom"     = amr_only[["Superkingdom"]],
  "Phylum"           = amr_only[["Phylum"]],
  "Taxonomic Class"  = amr_only[["Taxonomic Class"]],
  "Order"            = amr_only[["Order"]],
  "Family"           = amr_only[["Family"]],
  "Genus"            = amr_only[["Genus"]],
  "Species"          = amr_only[["Species"]],
  check.names = FALSE
)

wb2 <- createWorkbook()
addWorksheet(wb2, "AMR_Only")
writeData(wb2, "AMR_Only", amr_only)
freezePane(wb2, "AMR_Only", firstRow = TRUE)
setColWidths(wb2, "AMR_Only", cols = 1:ncol(amr_only), widths = "auto")
saveWorkbook(wb2, "A1_AMR_only.xlsx", overwrite = TRUE)

print(amr_only)